In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# ==============================
# 1. LOAD DATA
# ==============================
try:
    df = pd.read_csv('Chapter2/data/spam.csv', encoding='latin-1')[['v1','v2']]
except FileNotFoundError as e:
    raise FileNotFoundError("Không tìm thấy file spam.csv. Vui lòng kiểm tra đường dẫn: Chapter2/data/spam.csv") from e
df.columns = ['label','text']

# Encode label: ham = 0, spam = 1
df['label'] = df['label'].map({'ham':0,'spam':1})

# ==============================
# 2. PREPROCESSING
# ==============================
def preprocess(text):
    """
    - Lowercase
    - Remove special characters
    """
    return re.sub(r'[^a-z0-9\s]', '', text.lower())

df['text'] = df['text'].apply(preprocess)

# ==============================
# 3. TRAIN / TEST SPLIT
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42
)

# ==============================
# 4. TF-IDF VECTORIZATION
# ==============================
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

# ==============================
# 5. TRAIN SVM MODEL
# ==============================
"""
SVM tìm hyperplane để phân tách 2 lớp.

Kernel = 'linear':
- Phù hợp với text data (high-dimensional, sparse)
- Nhanh và hiệu quả hơn RBF trong bài toán NLP cơ bản
"""
model = SVC(kernel='linear')
model.fit(X_train, y_train)

# ==============================
# 6. EVALUATION
# ==============================
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ==============================
# 7. CONFUSION MATRIX (VISUALIZE)
# ==============================
cm = confusion_matrix(y_test, y_pred)

plt.figure()
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# ==============================
# 8. SAVE MODEL
# ==============================
joblib.dump(model, 'svm_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

print("Model saved!")

# ==============================
# 9. LOAD MODEL (DEMO)
# ==============================
loaded_model = joblib.load('svm_model.pkl')
loaded_vectorizer = joblib.load('tfidf_vectorizer.pkl')

# ==============================
# 10. TEST CUSTOM INPUT
# ==============================
msg = ["Congratulations! You won a free ticket"]
msg_clean = [preprocess(m) for m in msg]
msg_vec = loaded_vectorizer.transform(msg_clean)

pred = loaded_model.predict(msg_vec)

print("\nTest message prediction:", "Spam" if pred[0] == 1 else "Ham")


FileNotFoundError: Không tìm thấy file spam.csv. Vui lòng kiểm tra đường dẫn: Chapter2/data/spam.csv